# Lab 10 – Potoki ML i wyciek danych (data leakage)

*Czas: 2 × 45 min (część A + część B) · autograding: tak (A zalicza lab, B to rozszerzenie)*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2

# znajdź katalog główny repo (ten z folderem data/) – działa z labs/ i z submissions/<login>/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA = ROOT / "data"
print("DATA =", DATA)

**Układ zajęć:** Część A (~45 min) – wykrywanie przecieków i cechy. Część B (~45 min) – target encoding bez przecieku i `Pipeline`.

## Data leakage

In [ ]:
churn = pd.read_csv(DATA / "churn.csv")
churn.groupby("cancellation_reason")["churned"].mean()

`cancellation_reason` jest znana **dopiero po** odejściu klienta – ewidentny przeciek. **Typowe źródła:** (1) cechy z przyszłości, (2) skalowanie/imputacja na całych danych przed splitem, (3) target encoding bez separacji train/test, (4) duplikaty między train i test.

### Zadania A (autograding): `find_leaky_columns`, `add_features`
## Część B
### Zadania B (autograding): `target_encode`, `build_pipeline` (scikit-learn)

### Eksperyment: ile „daje” przeciek?

In [ ]:
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    base = pd.get_dummies(churn[["tenure_months", "support_calls", "contract"]], drop_first=True).astype(float)
    leak = base.join(pd.get_dummies(churn["cancellation_reason"]).astype(float))
    for name, X in [("uczciwe cechy", base), ("z przeciekiem", leak)]:
        auc = cross_val_score(LogisticRegression(max_iter=2000), X, churn["churned"], cv=5, scoring="roc_auc").mean()
        print(f"{name:15s} AUC = {auc:.3f}")
except ImportError:
    print("Zainstaluj scikit-learn")

## Pytania
1. Dlaczego imputację medianą trzeba robić w pipeline, a nie na całym zbiorze przed `train_test_split`?
2. Wygeneruj Copilotem „pełny pipeline dla churn”. Czy kod ma przeciek? Wskaż linię albo uzasadnij, że nie.

**Odpowiedzi:**
1. 
2. 

## Oddanie pracy

1. Zaimplementuj funkcje w pliku `exercises.py` (w tym samym folderze co ten notebook).
2. Sprawdź lokalnie: `SUBMISSION_DIR=submissions/<twój_login> pytest tests/test_lab10.py -v` (testy `test_a_*` = część A, `test_b_*` = część B)
   (Windows PowerShell: `$env:SUBMISSION_DIR="submissions/<twój_login>"; pytest tests/test_lab10.py -v`).
3. Skopiuj folder: `submissions/<twój_login>/lab10/` ma zawierać `exercises.py` i ten notebook (z odpowiedziami na pytania).
4. Otwórz Pull Request (szablon PR poprowadzi Cię dalej). CI uruchomi testy automatycznie.

> Szczegóły workflow: `CONTRIBUTING.md`.